# Planning Baseline

Switch the runtime to a GPU (**Runtime → Change runtime type → GPU**), then click **Runtime → Run all**.

This notebook downloads the simulator code, the frozen world model and policy, and the subgoal images, then runs validation scene `100038` twice: once with the policy alone and once with the baseline planner. With the default seed on our GPUs, the policy alone stacks one box and the planner stacks two; on a Colab T4 the result can differ slightly because it runs in float16. The whole notebook takes about 10 to 15 minutes.

After running it once, edit `GoalPlanningAgent` in the planner cell below to try your own planner.

In [ ]:
import os, subprocess, sys
os.environ["MUJOCO_GL"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["HF_HUB_DISABLE_XET"] = "1"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("installing system libraries", flush=True)
    env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
    subprocess.check_call(["apt-get", "update", "-qq"], env=env)
    subprocess.check_call(
        ["apt-get", "install", "-y", "-qq", "libegl1", "libgl1", "libgles2", "libglib2.0-0", "ffmpeg"],
        env=env,
    )

def missing(module):
    try:
        __import__(module)
        return False
    except Exception:
        return True

packages = []
if missing("mujoco"):
    packages.append("mujoco>=3.3,<4")
if missing("imageio_ffmpeg"):
    packages.append("imageio[ffmpeg]>=2.30")
if missing("huggingface_hub"):
    packages.append("huggingface_hub>=0.26")
if missing("matplotlib"):
    packages.append("matplotlib")
if packages:
    print("installing", ", ".join(packages), flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *packages])

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is visible. In Colab: Runtime → Change runtime type → GPU, then Runtime → Run all.")
print("cuda", torch.cuda.get_device_name(0), flush=True)


In [ ]:
import os, sys, tarfile
from pathlib import Path
os.environ["MUJOCO_GL"] = "egl"
os.environ["HF_HUB_DISABLE_XET"] = "1"

def find_root():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "assignment" / "training" / "planning" / "agent.py").exists():
            return cand
    return None

ROOT = find_root()
if ROOT is None:
    dest = Path("/content") if Path("/content").exists() else Path("/tmp")
    ROOT = dest / "world-model-gym"
    marker = ROOT / "assignment" / "training" / "planning" / "agent.py"
    if not marker.exists():
        print("downloading code", flush=True)
        from huggingface_hub import hf_hub_download
        archive = hf_hub_download("YongYong/cis6280-stack-world-models-v2", "code/world-model-gym.tar", repo_type="model")
        with tarfile.open(archive) as tar:
            try:
                tar.extractall(dest, filter="data")
            except TypeError:
                tar.extractall(dest)
    if not marker.exists():
        raise FileNotFoundError(f"code download did not produce {marker}")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
hand = ROOT / "assets" / "sharpa" / "wave_01" / "right_sharpa_wave" / "right_sharpa_wave.xml"
if not hand.exists():
    print("downloading hand assets", flush=True)
    __import__("subprocess").check_call([sys.executable, "scripts/fetch_assets.py"])
import assignment.training.planning.agent
print("code", ROOT, flush=True)

from assignment.training.data.task import StackTaskEnv, TaskConfig
_env = StackTaskEnv(TaskConfig(target_count=4, horizon=10, terminate_on_ground=False), 100038)
_env.reset(("init", 100038))
_env.env.resolution = 128
# render() returns the (front, side) camera tuple, each of shape (128, 128, 3).
_frames = list(_env.env.render())
_env.close()
print("render", _frames[0].shape, flush=True)

In [ ]:
import os, shutil
from pathlib import Path
os.environ["HF_HUB_DISABLE_XET"] = "1"
import torch
from huggingface_hub import hf_hub_download

CKPT = Path("/content/stack-baseline") if Path("/content").exists() else Path("/tmp/stack-baseline")
CKPT.mkdir(parents=True, exist_ok=True)
WM = CKPT / "smwm-c3.pt"
POLICY = CKPT / "df-smwm-c1-sigma04.pt"
GOALS = CKPT / "goals-every80.npz"

def fetch(repo, filename, dest, repo_type):
    if dest.exists() and dest.stat().st_size > 0:
        print("already downloaded", dest.name, flush=True)
        return dest
    print("downloading", filename, flush=True)
    src = hf_hub_download(repo, filename, repo_type=repo_type)
    shutil.copyfile(src, dest)
    return dest

fetch("YongYong/cis6280-stack-world-models-v2", "jepa/smwm-c3/model.pt", WM, "model")
fetch("YongYong/cis6280-stack-world-models-v2", "policy/df-smwm-c1-sigma04/policy.pt", POLICY, "model")
fetch("YongYong/cis6280-stack-datasets-v2", "goals/scripted-val100-every80/goals.npz", GOALS, "dataset")

# Point the policy checkpoint's encoder field to the local world-model file.
ckpt = torch.load(POLICY, map_location="cpu", weights_only=False)
ckpt["info"]["encoder"] = str(WM.resolve())
torch.save(ckpt, POLICY)
print("weights ready", WM.stat().st_size, POLICY.stat().st_size, GOALS.stat().st_size, flush=True)

## Subgoals

These are the subgoal images for scene `100038`, taken every 8 seconds from a scripted run. Each image shows the front wrist camera on the left and the side wrist camera on the right.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 100038
with np.load(GOALS) as goals:
    index = list(goals["seeds"]).index(SEED)
    count = int(goals["sub_count"][index])
    frames = goals["sub_frames"][index, :count]
    steps = goals["sub_steps"][index, :count]
shown = np.linspace(0, count - 1, num=min(count, 8), dtype=int)
fig, axes = plt.subplots(2, 4, figsize=(12, 4))
for ax, i in zip(axes.ravel(), shown):
    ax.imshow(frames[i])
    ax.set_title(f"step {int(steps[i])}", fontsize=9)
    ax.axis("off")
for ax in axes.ravel()[len(shown):]:
    ax.axis("off")
fig.tight_layout()
plt.show()
print(f"{count} subgoals, every 80 control steps")


## Baseline Planner (`GoalPlanningAgent`)

Without planning, `LatentPolicyAgent` samples one action chunk and executes its first block (5 control steps). With planning, `GoalPlanningAgent` samples 16 chunks, imagines each of them with the world model, and executes the first block of the chunk that ends closest to the current subgoal. It moves on to the next subgoal once the current frame is close enough to the active one.

To try your own planner for the competition, edit `GoalPlanningAgent` in the cell below and re-run the evaluation cell.

In [ ]:
import numpy as np
import torch
from assignment.training.data.cache import RAW_ACTION_DIM, STRIDE
from assignment.training.planning.agent import LatentPolicyAgent


class GoalPlanningAgent(LatentPolicyAgent):
    """Baseline test-time planner over policy action chunks using the latent world model.

    Edit this class to experiment with your own cost functions, rollout horizons,
    candidate refinement rules, or subgoal switching criteria.
    """

    def __init__(self, policy_path, device="cuda", candidates=16, tau=0.25, **kwargs):
        super().__init__(policy_path, device, **kwargs)
        self.candidates, self.tau = candidates, tau
        self.goals = None

    def reset(self, goal_frames=None, init_frame=None, seed=None):
        if goal_frames is not None and init_frame is None:
            raise ValueError("init_frame is required with goal_frames")
        super().reset(seed)
        self.frames, self.executed, self.index = [], [], 0
        self.last_distance = self.goals = self.gap = None
        if goal_frames is not None:
            # Encode all M subgoal frames (M, 128, 256, 3) -> (M, 192).
            self.goals = self._encode(np.asarray(goal_frames))
            prev = torch.cat([self._encode(np.asarray(init_frame)[None]), self.goals[:-1]])
            self.gap = (self.goals - prev).square().sum(-1)

    def goal_index(self, z):
        # Advance monotonically while ||z_k - g_m||_2^2 < tau * ||g_m - g_{m-1}||_2^2.
        while self.index < len(self.goals) - 1 and \
                (z - self.goals[self.index]).square().sum() < self.tau * self.gap[self.index]:
            self.index += 1
        return self.index

    @torch.no_grad()
    def _encode(self, frames):
        x = torch.from_numpy(np.ascontiguousarray(frames)).to(self.device)[:, None]
        with self._autocast():
            return self.encoder.encode(x)[:, 0].float()

    @torch.no_grad()
    def step(self, frame):
        if self.goals is None:
            raise RuntimeError("Call agent.reset(goal_frames, init_frame) before step().")
        # 1. Encode the current (128, 256, 3) frame into z_k in R^192 and update subgoal index.
        z = self._encode(frame[None])[0]
        ctx = self.encoder.cfg.context
        self.frames = (self.frames + [z])[-ctx:]
        self.history = (self.history + [(z - self.z_mean) / self.z_std])[-self.policy.cfg.context:]
        goal = self.goals[self.goal_index(z)]

        if not self.queue:
            # 2. Sample N = self.candidates action chunks of shape (N, generate, 125).
            pctx = [self.history[0]] * (self.policy.cfg.context - len(self.history)) + self.history
            n = self.candidates
            blocks = self.policy.sample(torch.stack(pctx)[None].expand(n, -1, -1), self.generate,
                                        self.n_steps, self.lag, self.generator)
            best = 0
            if n > 1:
                # 3. Roll all N candidates forward in the world model from the last C<=3 frames.
                past = self.executed[len(self.executed) - len(self.frames) + 1:] if len(self.frames) > 1 else []
                acts = torch.cat([torch.stack(past)[None].expand(n, -1, -1), blocks], 1) if past else blocks
                with self._autocast():
                    pred = self.encoder.rollout(torch.stack(self.frames)[None].expand(n, -1, -1),
                                                self.encoder.action_encoder(acts), self.generate)
                # 4. Pick the candidate whose final predicted latent is closest to the active subgoal.
                self.last_distance = (pred[:, -1].float() - goal).square().sum(-1)
                best = int(self.last_distance.argmin())
            raw = np.clip(blocks[best, :self.execute].cpu().numpy() * self.a_std + self.a_mean, -1, 1)
            self.queue = list(raw.reshape(-1, STRIDE, RAW_ACTION_DIM))

        block = self.queue.pop(0)
        self.executed.append(torch.from_numpy((block.reshape(-1) - self.a_mean) / self.a_std).to(self.device))
        return block

In [ ]:
import os, time
os.environ["MUJOCO_GL"] = "egl"
from pathlib import Path
import numpy as np

from assignment.training.data.cache import STRIDE
from assignment.training.data.recording import write_video
from assignment.training.data.signals import row_hand_parts
from assignment.training.data.task import StackTaskEnv, TaskConfig

HORIZON = 1795
OUT = Path("/content/stack-baseline") if Path("/content").exists() else Path("/tmp/stack-baseline")
OUT.mkdir(parents=True, exist_ok=True)
print("two runs, a few minutes each", flush=True)

def run(planning, video_path):
    with np.load(GOALS) as goals:
        i = list(goals["seeds"]).index(SEED)
        n = int(goals["sub_count"][i])
        goal_frames = goals["sub_frames"][i, :n].copy()
        init_frame = goals["init_frames"][i].copy()
    if planning:
        agent = GoalPlanningAgent(str(POLICY), "cuda", candidates=16, generate=16, execute=1,
                                  tau=0.25)
        agent.reset(goal_frames, init_frame, seed=SEED)
    else:
        agent = LatentPolicyAgent(str(POLICY), "cuda", generate=16, execute=1)
        agent.reset(seed=SEED)
    env = StackTaskEnv(TaskConfig(target_count=4, horizon=HORIZON, terminate_on_ground=False), SEED)
    env.reset(("init", SEED))
    env.env.resolution = 128
    frames, info, grasped, most = [], {}, False, 0
    started = time.time()
    for step in range(HORIZON):
        if step % STRIDE == 0:
            frame = np.concatenate(list(env.env.render()), axis=1)
            block = agent.step(frame)
            view = frame if not planning else np.concatenate(
                [frame, goal_frames[min(agent.index, len(goal_frames) - 1)]], axis=0)
            frames.append(view)
            if step % 200 == 0:
                print(f"{'planning' if planning else 'no planning'}  step {step:4d}  placed {most}", flush=True)
        _, _, terminated, truncated, info = env.step(block[step % STRIDE])
        most = max(most, int(info["placed"]))
        grasped = grasped or bool((row_hand_parts(env.env.history[-1])[0] >= 2).any())
        if terminated or truncated:
            break
    env.close()
    write_video(Path(video_path), [f[None] for f in frames], STRIDE)
    return dict(name="16 samples, subgoals, τ = 0.25" if planning else "one sample, no goal",
                placed=most, grasp=int(grasped),
                minutes=round((time.time() - started) / 60, 1))

rows = [run(False, OUT / "no-planning.mp4"), run(True, OUT / "planning.mp4")]
header = "| Selection | ≥1 | ≥2 | ≥3 | 4 | Grasp | minutes |\n|---|---|---|---|---|---|---|\n"
body = "\n".join(
    f"| {r['name']} | {int(r['placed']>=1)} | {int(r['placed']>=2)} | {int(r['placed']>=3)} | {int(r['placed']>=4)} | {r['grasp']} | {r['minutes']} |"
    for r in rows)
print("\nScene", SEED)
print(header + body)

In [ ]:
from IPython.display import Video, display, Markdown
display(Markdown("**No planning**"))
display(Video(str(OUT / "no-planning.mp4"), embed=True, width=384))
display(Markdown("**Planning.** The lower half of the frame is the current subgoal."))
display(Video(str(OUT / "planning.mp4"), embed=True, width=384))
display(Markdown("""
### 300 validation scenes, 3 sampler seeds

The cell above is one scene, so its 0/1 counts will not match this table. This is the number to beat.

| Selection | ≥1 | ≥2 | ≥3 | 4 | Grasp | Mean boxes |
|---|---|---|---|---|---|---|
| one sample, no goal | 81.4% | 35.7% | 8.6% | 1.1% | 99.4% | 1.27 |
| 16 samples, subgoals every 80 steps, τ = 0.25 | 85.2% | 41.4% | 11.8% | 2.4% | 99.1% | 1.41 |

`≥1` to `4` are the percentages of episodes whose stack reaches at least that many boxes at some point, and `Mean boxes` is the average peak number of stacked boxes.
"""))
